# Task 1 — Inductive Biases and Feature Representations (walkthrough)

**This notebook is OPTIONAL.** It does not contain any real logic of its own — it imports the same `.py` modules used by `scripts/run_task1.py` and runs the pipeline step-by-step with explanations. The canonical way to reproduce results is still:

```bash
python scripts/run_task1.py
```

Keeping all logic in the `.py` files (single source of truth) is why the notebook is safe to include alongside the suggested repository structure: nothing here can drift from the scripts.

> Reminder (AI Usage Policy): the **PDF report** must be written entirely by you. This notebook only produces numbers and figures.

## 0. Setup: make the project importable and load the config
We add the project root to the path and load `configs/config.yaml`, the single source of truth for every hyperparameter and seed.

In [ ]:
import os, sys
# When the notebook lives at the project root, that root is the current dir.
PROJECT_ROOT = os.path.abspath('.')
sys.path.insert(0, PROJECT_ROOT)

from utils import load_config, set_seed, get_device, ensure_dir
cfg = load_config()               # read every knob from configs/config.yaml
set_seed(cfg['seed'])             # seed 6304 -> full reproducibility
device = get_device()
print('device =', device)
print('seed   =', cfg['seed'])
print('dataset=', cfg['dataset']['name'])

## 1. Data: stratified split + class-balanced 500-image eval subset
`make_subset.py` fixes *which* images every model sees. The stratified split keeps class proportions in train/val; the eval subset is class-balanced and its identifiers are saved for reproducibility.

In [ ]:
from data.make_subset import (load_datasets, stratified_train_val_split,
                              build_eval_subset, save_identifiers)

train_ds, test_ds, class_names = load_datasets(cfg)   # downloads STL-10 on first run
train_view, val_view = stratified_train_val_split(train_ds, cfg)
eval_subset, sel_idx, imbalance = build_eval_subset(test_ds, cfg)
save_identifiers(sel_idx, imbalance, class_names, cfg)
print('train', len(train_view), '| val', len(val_view), '| eval', len(eval_subset))
print('classes:', class_names)

## 2. Models: frozen backbones + linear probe heads
Each backbone is frozen; we train only a linear head on cached features (a *linear probe*). CLIP additionally supports zero-shot classification with text prompts.

In [ ]:
from models.backbones import build_backbones, train_linear_head

backbones = build_backbones(cfg)
for bb in backbones.values():
    bb.to(device)

heads, head_info = {}, {}
for name, bb in backbones.items():
    print('training head for', name, '...')
    head, info = train_linear_head(bb, train_view, val_view, cfg)
    heads[name] = head
    head_info[name] = info
    print('   best val acc =', round(info['best_val_acc'], 4))

## 3. Run the full six-step pipeline
Rather than re-implement the six steps here, we call the orchestrator's `main()`. This guarantees the notebook and the script produce identical results. Use `--quick` semantics by editing the config if you want a fast test.

In [ ]:
# The cleanest option: run the whole thing exactly as the script does.
# (Comment this out if you prefer to step through experiments manually below.)
import subprocess, sys
print(subprocess.run([sys.executable, 'scripts/run_task1.py'],
                     capture_output=True, text=True).stdout)

## 4. Inspect results
All numbers are in `results/task1_results.json`; the compact comparison is in `results/summary_compact_comparison.csv`; figures are the `.png` files in `results/`.

In [ ]:
import json
with open(os.path.join(cfg['paths']['results_dir'], 'task1_results.json')) as f:
    res = json.load(f)
print('Clean baseline:')
for model, m in res['experiments']['clean_baseline'].items():
    print(f"  {model:16s} top1={m['top1']:.3f}  macroF1={m['macro_f1']:.3f}  meanMaxConf={m['mean_max_conf']:.3f}")

## 5. Where to go next
Open the `.png` figures and the JSON, then write your 8-page report and answer the four Research Questions **in your own words** (AI Usage Policy). This notebook has only produced the evidence.